# SparseBench Colab runner

Clones the repo, builds the CUDA extension, runs the tests.
No project code lives here. Edit on your laptop, push, then rerun these cells.

Needs a `GH_TOKEN` secret (🔑 panel on the left) with read access to the repo.

In [ ]:
# Which GPU did Colab give us? The build below targets A100 (compute_cap 8.0).
!nvidia-smi --query-gpu=name,compute_cap,driver_version --format=csv

In [ ]:
import base64
import pathlib
import subprocess

from google.colab import userdata

REPO = "saransh-debugs/SparseBench"
DEST = pathlib.Path("/content/SparseBench")


def git(*args: str) -> None:
    # The token goes in a header for this one command only. Putting it in the
    # clone URL would save it in .git/config and leak it in error messages.
    # Same reason we don't use check=True: its exception prints the full command.
    token = userdata.get("GH_TOKEN")
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    cmd = ["git", "-c", f"http.extraHeader=Authorization: Basic {auth}", *args]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout, result.stderr, sep="")
    if result.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed with exit code {result.returncode}")


if DEST.exists():
    git("-C", str(DEST), "pull", "--ff-only")
else:
    git("clone", f"https://github.com/{REPO}.git", str(DEST))
git("-C", str(DEST), "log", "--oneline", "-1")

In [ ]:
# Build for A100 only. Building 9.0a too would double compile time for no use here.
%cd /content/SparseBench
!TORCH_CUDA_ARCH_LIST="8.0" pip install --no-build-isolation -e ".[test]" -q

In [ ]:
!python -m pytest -q 2>&1 | tail -15

## Memory and race checks

`memcheck` catches out-of-bounds reads and writes. `racecheck` catches shared-memory races.
Skips the big tests (the sanitizer is ~100x slower) and the CUDA graph test (graphs need torch's memory cache, which is off here).

In [ ]:
# PyTorch's memory cache hands out slices of big blocks, so an out-of-bounds
# write can land in memory torch owns and memcheck won't see it. Turn it off,
# for these two commands only: torch disables the cache if this variable is
# set at all, even to 0.
!PYTORCH_NO_CUDA_MEMORY_CACHING=1 compute-sanitizer --tool memcheck --error-exitcode 1 python -m pytest tests/test_routing.py tests/test_dsd.py -q -k "not big and not graph and not D512" 2>&1 | tail -3
!PYTORCH_NO_CUDA_MEMORY_CACHING=1 compute-sanitizer --tool racecheck --error-exitcode 1 python -m pytest tests/test_routing.py tests/test_dsd.py -q -k "not big and not graph and not D512" 2>&1 | tail -3

## Benchmarks

In [ ]:
!python bench/bench_routing.py
!python bench/bench_dsd.py

## Hardware counters (ncu)

Counters only. ncu runs the GPU at base clock, so read times from the benchmark above, not from here.

In [ ]:
!bash bench/ncu.sh 1
!bash bench/ncu.sh 2